# Using PyVO to Find Observations for Analysis
<hr style="border: 2px solid #fadbac" />

- **Description:** A tutorial on using PyVO to find Obs IDs for Analysis.
- **Level:** Beginner
- **Data:** XMM observations of M 82 (obsid=Multiple)
- **Requirements:** Must be run using pySAS version 2.3.0 or higher.
- **Credit:** Ryan Tanner (February 2026)
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 15 February 2026, for SAS v22.1 and pySAS v2.3.0

<hr style="border: 2px solid #fadbac" />

## 1. Introduction

In this tutorial we will demonstrate how to use Virtual Observatory (VO) queries to create a list of Obs IDs of XMM observations that we can use for analysis. More information about the VO and what can be done with it can be found in the documentaiton for [NASA-NAVO Workshops](https://nasa-navo.github.io/navo-workshop/index.html).

We will use the VO to search for observations of M82 that we can use to generate images of the galaxy and the associated outflow.

#### SAS Tasks to be Used

None

#### Useful Links

- [`pysas` Documentation](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/pysas/index.html "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html "Helpdesk") - Link to form to contact the GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

In [ ]:
# Generic VO access routines
import pyvo as vo

# For specifying coordinates and angles
from astropy.coordinates import SkyCoord
from astropy.coordinates import Angle
from astropy import units as u

# Ignore unimportant warnings
import warnings
warnings.filterwarnings('ignore', '.*XDG_CACHE_HOME.*')

## 2. Search for XMM Regestries

<div class="alert alert-block alert-info">
    <b>Note:</b> This section assumes the user knows nothing about the Virtual Observatory (VO) and what registries are available. This section of the tutorial walks the user through several steps that are helpful to learn about the VO, but are not needed for those familiar with the VO.
</div>

### 2.1 Simple Search for Services

First we need to find databases or catalogs we can use to retrieve information about XMM observations. We will start by doing a search for all `registries` *worldwide* that contain XMM data. For this we use the function `regsearch` and look for `registries` with the keyword XMM.

In [ ]:
services = vo.regsearch(keywords=['xmm'])
services

Yikes! There are over 700 `registries` that contain XMM data. Let's try to narrow things down a bit. Let's add the keyword `heasarc` to limit it to results from the `HEASARC`.

In [ ]:
services = vo.regsearch(keywords=['xmm','heasarc'])
services

OK, we're down to only ~140 results. Let's pick one at random and find more information on it. Let's pick item number 42 in the list of services and display some information about this randomly chosen service. Some useful information that comes with the registry results are:

* `res_title` - A more descriptive title
* `short_name` - A short name
* `ivoid` - A unique identifier for the service.  Gives some indication of what organization is serving the data.
* `reference_url` - A link for more information
* `res_description` - A more verbose description

In [ ]:
s = services[41]

print(f'Registry Title      : {s.res_title}')
print(f'Short Name          : {s.short_name}')
print(f'ivoid               : {s.ivoid}')
print(f'reference_url       : {s.reference_url}')
print(f'Registry Description:\n    {s.res_description}')

***

OK. That's interesting, but not really something we can use. Rather than going through all 139 services individually to see if it possibly contains the information we are looking for, we can cheat and use the fact that we are looking ultimately for the master registry for XMM called `xmmmaster`.

In [ ]:
services = vo.regsearch(keywords=['xmm','heasarc'])
xmmmaster = [s for s in services if 'xmmmaster' in s.ivoid][0]

In [ ]:
print(f'Registry Title      : {xmmmaster.res_title}')
print(f'Short Name          : {xmmmaster.short_name}')
print(f'ivoid               : {xmmmaster.ivoid}')
print(f'reference_url       : {xmmmaster.reference_url}')
print(f'Registry Description:\n    {xmmmaster.res_description}')

### 2.2 Search the XMMMaster Registry for M82 Observations

In [ ]:
# This is just to make sure we have the right registry
services = vo.regsearch(keywords=['xmm','master','heasarc'])
xmmmaster = [s for s in services if 'xmmmaster' in s.ivoid][0]

Now we will search for observations within 0.1 degrees of M82. Let's get the coordinates of `M82`. The position can be found using a [SkyCoord](http://docs.astropy.org/en/stable/api/astropy.coordinates.SkyCoord.html) object.

In [ ]:
m82_pos = SkyCoord.from_name('m82')
m82_pos

The size of the region is specified with the `radius` keyword and may be decimal degrees or an [Astropy Angle](http://docs.astropy.org/en/stable/api/astropy.coordinates.Angle.html#astropy.coordinates.Angle).

In [ ]:
results = xmmmaster.search(pos=m82_pos, radius=0.1)
# Astropy Table is useful for displaying search results.
table_of_results = results.to_table()
table_of_results

## 3. Searching Using a TAP Service

Now we have a list of 20 potential Obs IDs. But right now we have no idea if any of these Obs IDs are even useful. We need a more powerful search to check the observations for a few things before we start downloading data. For that we use the HEASARC `Table Access Protocol` (`TAP`) service. First we use `regsearch` to find the `TAP` service for the HEASARC. The `TAP` service is an *alternative* to what we used in the previous section.

In [ ]:
tap_services = vo.regsearch(servicetype='tap',keywords=['heasarc'])
print(f'{len(tap_services)} service found.')
heasarc_tap = tap_services[0]
print(f'{heasarc_tap.describe()}\n')

This is the `TAP` service for the *entire* HEASARC, which means it has **a lot** of stuff other than just XMM information. Let's take a look at all the options. There should be 1,000+ differnt tables.

In [ ]:
tables = heasarc_tap.service.tables  # Queries for details of the service's tables
print(f'{len(tables)} tables')
for t in tables:
    print(f'{t.name:30s} - {t.description}')  # A more succinct option than t.describe()

Again, what we need is the `xmmmaster` table.

In [ ]:
xmmmaster_table = tables['xmmmaster']

Let's look at what is available in the `xmmmaster` TAP table.

In [ ]:
for c in xmmmaster_table.columns:
    print(f'{c.name:20s} - {c.description}')

***

All of the columns above are things that we can use to generate specific searches to filter potential observations. We can do this by formatting our query in a language called [`ADQL`](http://www.ivoa.net/documents/latest/ADQL.html), which is based on `SQL`. The basic form of an `ADQL` query is:

```
"""SELECT [columns you want to select] FROM [name of TAP table] WHERE [filtering conditions to be met]"""
```

For example, if you wanted to return **all** information related to **all** Obs IDs in `xmmmaster` you would use (WARNING: DON'T DO THIS!!!):

```
"""SELECT * FROM xmmmaster"""
```

(SERIOUSLY. DON'T DO THIS. At best your request will time out. At worst you might crash *your* computer.)

We need a way of narrowing down our search to just the observations that fall within 0.1 degrees of M82. `ADQL` queries have built in functions to help us with that. That function looks like this:

```
contains(point('ICRS',[table name].ra,[table name].dec),circle('ICRS',[target RA],[target DEC],0.1))=1
```

Searching for observations around M82 would look like this:

```
"""SELECT * FROM xmmmaster WHERE contains(point('ICRS',xmmmaster.ra,xmmmaster.dec),circle('ICRS',148.9684583,69.6797028,0.1))=1"""
```

We can format the string to automatically insert the position of our target.

```python
"""SELECT * FROM xmmmaster WHERE contains(point('ICRS',xmmmaster.ra,xmmmaster.dec),circle('ICRS',{},{},0.1))=1""".format(m82_pos.ra.deg, m82_pos.dec.deg)
```

Let's try this.

In [ ]:
m82_pos = SkyCoord.from_name('m82')
query = """SELECT * FROM xmmmaster WHERE contains(point('ICRS',xmmmaster.ra,xmmmaster.dec),circle('ICRS',{},{},0.1))=1""".format(m82_pos.ra.deg, m82_pos.dec.deg)
tab = heasarc_tap.search(query).to_table()
tab

Let's narrow down which columns are returned. We will restrict it to `obsid`, `mos1_mode`, `mos2_mode`, and `pn_mode`.

In [ ]:
query = """SELECT obsid, mos1_mode, mos2_mode, pn_mode FROM xmmmaster WHERE contains(point('ICRS',xmmmaster.ra,xmmmaster.dec),circle('ICRS',{},{},0.1))=1""".format(m82_pos.ra.deg, m82_pos.dec.deg)
tab = heasarc_tap.search(query).to_table()
tab

***

### EPIC Modes and Filters

Looking at this we see that there are four observations where the EPIC cameras were not used. For the EPIC cameras there are five different operating modes for both the MOS and pn, with a few variations for the "Small Window" mode for the MOS, and the "Full Frame" mode for the pn. The number in parentheses is the number of exposures for that observation.

#### MOS Modes

- (FF): Full Frame
- (LW): Large Window
- (SW): Small Window
    - (SWn): Small Window (CCD number: n = 2, 3, 4, 5, or 6)
    - (SW RFS): Small Window Refresh Frame Store
- (FU): Timing Uncompressed
- (FC): Timing Compressed

#### pn Modes

- (FF): Full Frame
    - (FLG): Full Frame Low Gain
    - (EFF): Extended Full Frame
    - (FMK): Full Frame Masked
- (LW): Large window
- (SW): Small window
- (TI): Timing
- (BU): Burst

#### EPIC Filters

- (TK): Thick
- (ME): Medium
- (TN): Thin

### RGS Modes

- (SPE): Spectroscopy
- (SSW): Spectroscopy Small Window
- (SES): Single Event Selection
- (HER): High Event Rate
- (SER): Split Event Reconstruction

### Optical Monitor Modes and Filters

#### OM Modes

- (FA): Fast
- (IM): Image
- (IF): Image + Fast

#### OM Filters

- (U) : U-band Filter
- (B) : B-band Filter
- (V) : V-band Filter
- (W) : White Filter
- (M2): UVM2 Filter
- (W2): UVW2 Filter
- (W1): UVW1 Filter
- (GRISM2): Optical Grism
- (GRISM1): UV Grism

Now we want to filter the `TAP` data based on mode and filter. We want results that used Full Frame mode and the medium filter. The query would look like this:

In [ ]:
query = """SELECT obsid, mos1_mode, mos2_mode, pn_mode
           FROM xmmmaster 
           WHERE xmmmaster.mos1_mode LIKE 'FF-ME%' AND 
                 xmmmaster.mos2_mode LIKE 'FF-ME%' AND 
                 xmmmaster.pn_mode LIKE 'FF-ME%' AND 
                 contains(point('ICRS',xmmmaster.ra,xmmmaster.dec),circle('ICRS',{},{},0.1))=1
        """.format(m82_pos.ra.deg, m82_pos.dec.deg)
tab = heasarc_tap.search(query).to_table()
tab

This uses a wildcard (`%`) in the search query to match all results that contain the expression `FF-ME`. More information on how to use this can be found by looking up information on the SQL `LIKE` Operator.

### Some Additional Notes:

There is some flexability in how the query is written. For example, we can shorten the name of the `TAP` table in the query using,

```
FROM xmmmaster AS CAT
```

For example,

In [ ]:
query = """SELECT obsid, mos1_mode, mos2_mode, pn_mode
           FROM xmmmaster AS CAT 
           WHERE CAT.mos1_mode LIKE 'FF-ME%' AND 
                 CAT.mos2_mode LIKE 'FF-ME%' AND 
                 CAT.pn_mode LIKE 'FF-ME%' AND 
                 contains(point('ICRS',CAT.ra,CAT.dec),circle('ICRS',{},{},0.1))=1
        """.format(m82_pos.ra.deg, m82_pos.dec.deg)
tab = heasarc_tap.search(query).to_table()
tab

The query is not case sensitive. For example,

In [ ]:
query = """selEcT ObsId, mos1_moDe, MOS2_MODE, pN_mode
           fRoM xMmMAsTeR as cat 
           WhErE CAT.mos1_mode LIKE 'FF-ME%' and 
                 cAt.mos2_mode LIKE 'FF-ME%' aNd 
                 caT.pn_mode LIKE 'FF-ME%' anD 
                 contains(point('ICRS',CAt.ra,cAT.dec),circle('ICRS',{},{},0.1))=1
        """.format(m82_pos.ra.deg, m82_pos.dec.deg)
tab = heasarc_tap.search(query).to_table()
tab

You can also rename the `TAP` table whatever you want in your search (i.e. it doesn't have to be `cat`).

In [ ]:
query = """SELECT obsid, mos1_mode, mos2_mode, pn_mode
           FROM xmmmaster AS DOG 
           WHERE DOG.mos1_mode LIKE 'FF-ME%' AND 
                 DOG.mos2_mode LIKE 'FF-ME%' AND 
                 DOG.pn_mode LIKE 'FF-ME%' AND 
                 contains(point('ICRS',DOG.ra,DOG.dec),circle('ICRS',{},{},0.1))=1
        """.format(m82_pos.ra.deg, m82_pos.dec.deg)
tab = heasarc_tap.search(query).to_table()
tab